# Qubits: Amplitudes, Phases and the Bloch Sphere

In every earlier application, complex numbers were a clever shortcut: a phasor or a complex amplitude stood in for a real cosine, and at the end we took the real part. Quantum mechanics is different. The state of a quantum bit is a pair of **complex amplitudes** $(\alpha, \beta)$, and there is no "real part" to take at the end: the phases are physical. A measurement only reveals the moduli $|\alpha|^2$ and $|\beta|^2$, yet the phases decide what happens when amplitudes are combined — **interference**.

This notebook builds the qubit from complex arithmetic: normalization, global versus relative phase, the Bloch sphere parametrisation, quantum gates as unitary $2 \times 2$ complex matrices, and an interference experiment in which changing nothing but a phase changes the measurement statistics from certain to impossible. Everything is plain numpy.

This is notebook 06 of module 07 (**Applications**) in the **Learning Complex Numbers** series.

## 1. Introduction

A classical bit is $0$ or $1$. A **qubit** — the spin of an electron, the polarization of a photon, two energy levels of an atom — is described by a unit vector in $\mathbb{C}^2$:

$$
|\psi\rangle = \alpha|0\rangle + \beta|1\rangle, \qquad \alpha, \beta \in \mathbb{C}.
$$

Measuring it gives $0$ with probability $|\alpha|^2$ and $1$ with probability $|\beta|^2$. If that were all, the amplitudes could be real, or even replaced by probabilities. The complex numbers matter because quantum evolution acts on **amplitudes**, which add like arrows in the complex plane before they are squared. Two arrows of length $\frac12$ can add to length $1$ or cancel to $0$ depending only on the angle between them — that angle is a phase.

Everything in this notebook is complex arithmetic from earlier modules: modulus and conjugate (00/03), polar form and $e^{i\varphi}$ (01/01, 01/03), multiplication as rotation (01/02), and unitary matrices (04/04).

**Prerequisites**

- Polar form and Euler's formula — [Polar form, modulus and argument](../01_polar_form_and_euler/01_polar_form_modulus_and_argument.ipynb), [Euler's formula](../01_polar_form_and_euler/03_eulers_formula.ipynb).
- Complex vectors and the inner product $\langle u, v\rangle = \sum \bar u_k v_k$ — [Complex vectors and inner products](../04_complex_linear_algebra/03_complex_vectors_and_inner_products.ipynb).
- Hermitian and unitary matrices — [Hermitian and unitary matrices](../04_complex_linear_algebra/04_hermitian_and_unitary_matrices.ipynb).
- The extended plane and the Riemann sphere (for the stereographic view of the Bloch sphere) — [Möbius transformations](../03_complex_functions/06_mobius_transformations.ipynb).
- From the quantum-computing course — [Complex linear algebra](../../../quantum-computing-course/notebooks/00_complex_linear_algebra.ipynb) and [The qubit](../../../quantum-computing-course/notebooks/01_the_qubit.ipynb).

After completing this notebook you will be able to:

- Normalize a qubit state and compute measurement probabilities with the Born rule.
- Distinguish a **global** phase (unobservable) from a **relative** phase (observable).
- Convert a state to Bloch-sphere angles $(\theta, \varphi)$ and a Bloch vector, and back.
- Write the Pauli, Hadamard, $S$ and $T$ gates, check that they are unitary, and describe their action on the Bloch sphere.
- Predict and simulate an interference experiment where a phase controls the measurement probabilities.

## 2. Theory

### 2.1 State vectors and the Born rule

With the basis vectors $|0\rangle = \begin{pmatrix}1\\0\end{pmatrix}$ and $|1\rangle = \begin{pmatrix}0\\1\end{pmatrix}$, a qubit state is a column vector $|\psi\rangle = \begin{pmatrix}\alpha\\\beta\end{pmatrix}$ with

$$
\langle\psi|\psi\rangle = |\alpha|^2 + |\beta|^2 = 1.
$$

($\langle\psi|$ is the conjugate transpose, the row $(\bar\alpha, \bar\beta)$.) **Born rule:** measuring in the basis $\{|0\rangle, |1\rangle\}$ gives outcome $0$ with probability $|\alpha|^2$ and $1$ with probability $|\beta|^2$. More generally, measuring in any orthonormal basis $\{|e_0\rangle, |e_1\rangle\}$ gives outcome $k$ with probability $|\langle e_k|\psi\rangle|^2$. Two other bases recur constantly:

$$
|\pm\rangle = \frac{|0\rangle \pm |1\rangle}{\sqrt2}, \qquad |\pm i\rangle = \frac{|0\rangle \pm i|1\rangle}{\sqrt2}.
$$

### 2.2 Global phase versus relative phase

**Global phase.** Multiplying the whole state by $e^{i\gamma}$ changes nothing observable: for every basis vector, $|\langle e|e^{i\gamma}\psi\rangle|^2 = |e^{i\gamma}|^2\,|\langle e|\psi\rangle|^2 = |\langle e|\psi\rangle|^2$. So $|\psi\rangle$ and $e^{i\gamma}|\psi\rangle$ are the **same physical state**.

**Relative phase.** Write $\alpha = |\alpha|e^{i a}$, $\beta = |\beta|e^{i b}$. Pulling out $e^{ia}$ as a global phase leaves $|\alpha|\,|0\rangle + |\beta|e^{i(b - a)}|1\rangle$. The difference $\varphi = b - a$ is the **relative phase**, and it *is* observable. Example: $|+\rangle$ and $|-\rangle$ differ only by the relative phase $\pi$; both give $0$ or $1$ with probability $\frac12$ in the standard basis, yet in the $\pm$ basis one gives "$+$" with certainty and the other never does.

So a qubit has two real degrees of freedom: from four real numbers in $(\alpha, \beta)$, normalization removes one and the global phase another.

### 2.3 The Bloch sphere

Choose the global phase so that $\alpha$ is real and non-negative. Normalization then lets us write

$$
|\psi\rangle = \cos\frac{\theta}{2}\,|0\rangle + e^{i\varphi}\sin\frac{\theta}{2}\,|1\rangle, \qquad \theta \in [0, \pi],\ \varphi \in [0, 2\pi),
$$

and $(\theta, \varphi)$ are the polar and azimuthal angles of a point on the unit sphere, the **Bloch vector**

$$
\vec r = (\sin\theta\cos\varphi,\ \sin\theta\sin\varphi,\ \cos\theta) = \left(2\,\mathrm{Re}(\bar\alpha\beta),\ 2\,\mathrm{Im}(\bar\alpha\beta),\ |\alpha|^2 - |\beta|^2\right).
$$

(The right-hand form works for any global phase, because $\bar\alpha\beta$ is unchanged by $e^{i\gamma}$.) Landmarks: $|0\rangle$ is the north pole, $|1\rangle$ the south pole, $|\pm\rangle$ are $\pm x$, $|\pm i\rangle$ are $\pm y$. The **relative phase is the longitude**, and the balance $|\alpha|^2 : |\beta|^2$ sets the latitude.

Note the half angle $\theta/2$: orthogonal states, which are $90°$ apart in $\mathbb{C}^2$, sit at **antipodal** points of the sphere.

*Stereographic view.* The ratio $w = \beta/\alpha = e^{i\varphi}\tan\frac{\theta}{2} \in \mathbb{C} \cup \{\infty\}$ is also unchanged by a global phase. It is exactly the stereographic projection of the Bloch vector from the south pole onto the equatorial plane, so the Bloch sphere *is* the Riemann sphere of notebook 03/06, with $|0\rangle \leftrightarrow 0$ and $|1\rangle \leftrightarrow \infty$.

### 2.4 Gates are unitary matrices

Closed quantum evolution is linear and preserves total probability, so it is given by a matrix $U$ with $\|U\psi\| = \|\psi\|$ for all $\psi$, i.e. a **unitary** matrix: $U^\dagger U = I$. Its columns are orthonormal, $|\det U| = 1$, and its eigenvalues lie on the unit circle. The standard single-qubit gates:

$$
X = \begin{pmatrix}0 & 1\\ 1 & 0\end{pmatrix},\quad
Y = \begin{pmatrix}0 & -i\\ i & 0\end{pmatrix},\quad
Z = \begin{pmatrix}1 & 0\\ 0 & -1\end{pmatrix},\quad
H = \frac{1}{\sqrt2}\begin{pmatrix}1 & 1\\ 1 & -1\end{pmatrix},
$$

$$
R(\phi) = \begin{pmatrix}1 & 0\\ 0 & e^{i\phi}\end{pmatrix},\qquad S = R\!\left(\tfrac{\pi}{2}\right) = \begin{pmatrix}1 & 0\\ 0 & i\end{pmatrix},\qquad T = R\!\left(\tfrac{\pi}{4}\right) = \begin{pmatrix}1 & 0\\ 0 & e^{i\pi/4}\end{pmatrix}.
$$

On the Bloch sphere every single-qubit gate is a **rotation**:

- $X$, $Y$, $Z$: half-turns ($180°$) about the $x$, $y$, $z$ axes. $X$ is the quantum NOT: $X|0\rangle = |1\rangle$.
- $R(\phi)$ multiplies $\beta$ by $e^{i\phi}$: it adds $\phi$ to the relative phase, i.e. rotates by $\phi$ about the $z$-axis. $T^2 = S$, $S^2 = Z$: an eighth, a quarter and a half turn.
- $H$ is a half-turn about the diagonal axis $(x + z)/\sqrt2$; it swaps $z \leftrightarrow x$: $H|0\rangle = |+\rangle$, $H|+\rangle = |0\rangle$, $H|-\rangle = |1\rangle$, and $HZH = X$.

Phase gates are invisible to an immediate standard-basis measurement (they leave $|\alpha|$ and $|\beta|$ alone). They matter only through what happens next.

### 2.5 Interference

Sandwich a phase gate between two Hadamards, starting from $|0\rangle$:

$$
|0\rangle \xrightarrow{\ H\ } \frac{|0\rangle + |1\rangle}{\sqrt2} \xrightarrow{\ R(\phi)\ } \frac{|0\rangle + e^{i\phi}|1\rangle}{\sqrt2} \xrightarrow{\ H\ } \frac{1 + e^{i\phi}}{2}|0\rangle + \frac{1 - e^{i\phi}}{2}|1\rangle.
$$

The amplitude for outcome $0$ is the **sum of two arrows**, $\frac12$ and $\frac12e^{i\phi}$ — one for each "path" through the intermediate states $|0\rangle$ and $|1\rangle$. Hence

$$
P(0) = \left|\frac{1 + e^{i\phi}}{2}\right|^2 = \frac{2 + 2\cos\phi}{4} = \cos^2\frac{\phi}{2}, \qquad P(1) = \sin^2\frac{\phi}{2}.
$$

At $\phi = 0$ the arrows align (constructive interference, $P(0) = 1$); at $\phi = \pi$ they cancel (destructive, $P(0) = 0$). If you instead measured after the first $H$, each path would be taken with probability $\frac12$ and adding *probabilities* would give $P(0) = \frac12$ for every $\phi$. Adding amplitudes, not probabilities, is the essence of quantum mechanics — this circuit is a Mach–Zehnder interferometer, and phase-controlled interference of this kind drives the quantum Fourier transform and phase estimation.

## 3. Worked Examples

### Example 1 — normalizing and measuring

Normalize $(1 + i)|0\rangle + 2|1\rangle$. Its squared norm is $|1 + i|^2 + |2|^2 = 2 + 4 = 6$, so

$$
|\psi\rangle = \frac{(1 + i)|0\rangle + 2|1\rangle}{\sqrt6}, \qquad P(0) = \frac{2}{6} = \frac13, \quad P(1) = \frac46 = \frac23.
$$

### Example 2 — Bloch angles of that state

1. $\alpha = \frac{1 + i}{\sqrt6} = \frac{1}{\sqrt3}e^{i\pi/4}$. Remove the global phase $e^{i\pi/4}$: $\alpha' = \frac{1}{\sqrt3}$, $\beta' = \frac{2}{\sqrt6}e^{-i\pi/4} = \sqrt{\tfrac23}\,e^{-i\pi/4}$.
2. $\cos\frac\theta2 = \frac{1}{\sqrt3}$, so $\theta = 2\arccos\frac{1}{\sqrt3} = 1.9106$ rad ($109.47°$); $\varphi = -\frac\pi4 \equiv \frac{7\pi}{4}$.
3. Bloch vector via $\bar\alpha\beta = \frac{(1 - i)\cdot 2}{6} = \frac{1 - i}{3}$: $\vec r = \left(\frac23, -\frac23, \frac13 - \frac23\right) = \left(\frac23, -\frac23, -\frac13\right)$, a unit vector ✓. Southern hemisphere (more likely to give $1$), longitude $-45°$.

### Example 3 — a global phase in disguise

$\frac{i|0\rangle - |1\rangle}{\sqrt2} = i\cdot\frac{|0\rangle + i|1\rangle}{\sqrt2} = i\,|{+i}\rangle$. Same physical state as $|{+i}\rangle$: the $+y$ point of the sphere.

### Example 4 — relative phase is visible in another basis

$|+\rangle$ and $|-\rangle$ both give $P(0) = P(1) = \frac12$. In the $\pm$ basis: $\langle +|+\rangle = 1$, $\langle +|-\rangle = \frac{1 - 1}{2} = 0$. Equivalently, apply $H$ and measure in the standard basis: $H|+\rangle = |0\rangle$, $H|-\rangle = |1\rangle$ — perfectly distinguishable.

### Example 5 — $HZH = X$

$$
HZ = \frac{1}{\sqrt2}\begin{pmatrix}1 & -1\\ 1 & 1\end{pmatrix}, \qquad HZH = \frac12\begin{pmatrix}1 & -1\\ 1 & 1\end{pmatrix}\begin{pmatrix}1 & 1\\ 1 & -1\end{pmatrix} = \frac12\begin{pmatrix}0 & 2\\ 2 & 0\end{pmatrix} = X.
$$

On the sphere: $H$ swaps the $x$ and $z$ axes, so a half-turn about $z$ seen through $H$ is a half-turn about $x$.

### Example 6 — interference with $S$ and $T$

In $H\,R(\phi)\,H|0\rangle$: with $S$ ($\phi = \frac\pi2$) the amplitudes are $\frac{1 + i}{2}$ and $\frac{1 - i}{2}$, each of modulus $\frac{1}{\sqrt2}$, so $P(0) = \frac12$. With $T$ ($\phi = \frac\pi4$): $P(0) = \cos^2\frac\pi8 = \frac{2 + \sqrt2}{4} \approx 0.854$. With $Z$ ($\phi = \pi$): $P(0) = 0$ — the circuit is $HZH = X$, a NOT.

## 4. Python Examples

### 4.1 States, normalization and the Born rule

In [ ]:
import numpy as np

ket0 = np.array([1, 0], dtype=complex)
ket1 = np.array([0, 1], dtype=complex)


def normalize(state):
    return state / np.linalg.norm(state)


def probabilities(state, basis=None):
    # Born rule; basis is a list of orthonormal vectors (default: |0>, |1>)
    if basis is None:
        basis = [ket0, ket1]
    return np.array([abs(np.vdot(e, state)) ** 2 for e in basis])     # np.vdot conjugates its first argument


psi = normalize((1 + 1j) * ket0 + 2 * ket1)
print("psi =", np.round(psi, 4), "  norm =", round(np.linalg.norm(psi), 12))
print("P(0), P(1) =", np.round(probabilities(psi), 6), " (1/3, 2/3)")

plus, minus = normalize(ket0 + ket1), normalize(ket0 - ket1)
plus_i, minus_i = normalize(ket0 + 1j * ket1), normalize(ket0 - 1j * ket1)
for name, state in [("|+>", plus), ("|->", minus), ("|+i>", plus_i)]:
    print(f"{name:5s} standard basis {np.round(probabilities(state), 3)}   +/- basis {np.round(probabilities(state, [plus, minus]), 3)}")

### 4.2 Global phase changes nothing, relative phase does

For a random state, multiply by $e^{i\gamma}$ and compare the probabilities in three bases; then change only the relative phase.

In [ ]:
import numpy as np

ket0, ket1 = np.array([1, 0], dtype=complex), np.array([0, 1], dtype=complex)
bases = {
    "Z": [ket0, ket1],
    "X": [(ket0 + ket1) / np.sqrt(2), (ket0 - ket1) / np.sqrt(2)],
    "Y": [(ket0 + 1j * ket1) / np.sqrt(2), (ket0 - 1j * ket1) / np.sqrt(2)],
}


def probabilities(state, basis):
    return np.array([abs(np.vdot(e, state)) ** 2 for e in basis])


rng = np.random.default_rng(7)
psi = rng.normal(size=2) + 1j * rng.normal(size=2)
psi /= np.linalg.norm(psi)
globally_shifted = np.exp(1.234j) * psi
relatively_shifted = np.array([psi[0], np.exp(1.234j) * psi[1]])

for name, basis in bases.items():
    print(f"{name} basis: psi {np.round(probabilities(psi, basis), 4)}   e^(i g) psi {np.round(probabilities(globally_shifted, basis), 4)}"
          f"   relative phase changed {np.round(probabilities(relatively_shifted, basis), 4)}")

### 4.3 Bloch angles and Bloch vectors

`bloch_angles` removes the global phase by rotating $\alpha$ onto the positive real axis; `bloch_vector` uses the phase-free formula $\left(2\mathrm{Re}(\bar\alpha\beta), 2\mathrm{Im}(\bar\alpha\beta), |\alpha|^2 - |\beta|^2\right)$. Both agree, and the round trip angles → state → angles is exact up to a global phase.

In [ ]:
import numpy as np


def bloch_angles(state):
    alpha, beta = state / np.linalg.norm(state)
    theta = 2 * np.arccos(np.clip(abs(alpha), 0, 1))
    if abs(alpha) < 1e-12 or abs(beta) < 1e-12:
        return theta, 0.0                                   # at a pole the longitude is undefined
    phi = (np.angle(beta) - np.angle(alpha)) % (2 * np.pi)
    return theta, phi


def bloch_vector(state):
    alpha, beta = state / np.linalg.norm(state)
    cross = np.conj(alpha) * beta
    return np.array([2 * cross.real, 2 * cross.imag, abs(alpha) ** 2 - abs(beta) ** 2])


def state_from_angles(theta, phi):
    return np.array([np.cos(theta / 2), np.exp(1j * phi) * np.sin(theta / 2)])


psi = np.array([1 + 1j, 2]) / np.sqrt(6)
theta, phi = bloch_angles(psi)
print(f"Ex 2: theta = {theta:.4f} rad ({np.degrees(theta):.2f} deg), phi = {phi:.4f} rad (7pi/4 = {7 * np.pi / 4:.4f})")
print("Bloch vector:", np.round(bloch_vector(psi), 6))
print("from angles :", np.round([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)], 6))

rebuilt = state_from_angles(theta, phi)
overlap = abs(np.vdot(rebuilt, psi))
print("|<rebuilt|psi>| =", round(overlap, 12), " -> same state up to the global phase", np.round(np.vdot(rebuilt, psi), 4))

### 4.4 Gates: unitarity, identities and eigenvalues on the unit circle

In [ ]:
import numpy as np

X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.array([[1, 0], [0, -1]], dtype=complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
S = np.diag([1, 1j])
T = np.diag([1, np.exp(1j * np.pi / 4)])
gates = {"X": X, "Y": Y, "Z": Z, "H": H, "S": S, "T": T}

for name, U in gates.items():
    unitary = np.allclose(U.conj().T @ U, np.eye(2))
    eigenvalues = np.linalg.eigvals(U)
    determinant = U[0, 0] * U[1, 1] - U[0, 1] * U[1, 0]
    print(f"{name}: unitary {unitary}, |det| = {abs(determinant):.3f}, eigenvalues {np.round(eigenvalues, 4)}, |eig| = {np.round(np.abs(eigenvalues), 6)}")

print()
print("T^2 == S:", np.allclose(T @ T, S), "  S^2 == Z:", np.allclose(S @ S, Z), "  HZH == X:", np.allclose(H @ Z @ H, X),
      "  HXH == Z:", np.allclose(H @ X @ H, Z), "  XY == iZ:", np.allclose(X @ Y, 1j * Z))

### 4.5 Simulating measurements: the interference experiment

Measurement outcomes are random. We sample $1000$ "shots" of the circuit $H\,R(\phi)\,H$ on $|0\rangle$ for several phases and compare the observed frequencies with $\cos^2(\phi/2)$.

In [ ]:
import numpy as np

H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
ket0 = np.array([1, 0], dtype=complex)
rng = np.random.default_rng(2024)
shots = 1000


def phase_gate(phi):
    return np.diag([1, np.exp(1j * phi)])


print(f"{'phi':>8s} {'amp(0)':>18s} {'P(0) exact':>11s} {'cos^2(phi/2)':>13s} {'observed':>9s}")
for phi in [0, np.pi / 4, np.pi / 2, 2 * np.pi / 3, np.pi, 3 * np.pi / 2]:
    final = H @ phase_gate(phi) @ H @ ket0
    p0 = abs(final[0]) ** 2
    outcomes = rng.choice([0, 1], size=shots, p=[p0, 1 - p0])
    observed = np.mean(outcomes == 0)
    print(f"{phi:8.4f} {np.round(final[0], 4) + 0!s:>18s} {p0:11.4f} {np.cos(phi / 2) ** 2:13.4f} {observed:9.3f}")

## 5. Visualizations

### 5.1 Amplitudes as arrows: the relative phase is the angle between them

Each state is drawn as its two amplitudes in the complex plane ($\alpha$ blue, $\beta$ orange). The four states $|+\rangle$, $|-\rangle$, $|{+i}\rangle$ and $e^{i\pi/3}|+\rangle$ all have $|\alpha| = |\beta| = \frac{1}{\sqrt2}$ and identical standard-basis statistics. What distinguishes the first three is the angle *between* the arrows (the relative phase $0$, $\pi$, $\frac\pi2$); the fourth is the first rotated as a whole (a global phase), the same physical state as $|+\rangle$.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

s = 1 / np.sqrt(2)
states = [
    ("$|+\\rangle$", np.array([s, s]), "0"),
    ("$|-\\rangle$", np.array([s, -s]), "$\\pi$"),
    ("$|+i\\rangle$", np.array([s, 1j * s]), "$\\pi/2$"),
    ("$e^{i\\pi/3}|+\\rangle$", np.exp(1j * np.pi / 3) * np.array([s, s]), "0"),
]

fig, axes = plt.subplots(1, 4, figsize=(16, 4.4))
circle = np.exp(1j * np.linspace(0, 2 * np.pi, 200))
for ax, (title, (alpha, beta), relative) in zip(axes, states):
    ax.plot(circle.real, circle.imag, color="lightgray")
    # alpha drawn wide and pale underneath, beta narrow on top, so coinciding arrows stay visible
    ax.annotate("", xy=(alpha.real, alpha.imag), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="tab:blue", linewidth=6, alpha=0.45))
    ax.annotate("", xy=(beta.real, beta.imag), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="tab:orange", linewidth=2))
    for value, color, label, side in [(alpha, "tab:blue", "$\\alpha$", -1), (beta, "tab:orange", "$\\beta$", 1)]:
        normal = 1j * value / abs(value)          # perpendicular offset keeps the two labels apart
        position = 1.2 * value + side * 0.13 * normal
        ax.text(position.real, position.imag, label, color=color, fontsize=13, ha="center", va="center")
    ax.set_title(f"{title}: relative phase {relative}")
    ax.axhline(0, color="black", linewidth=0.8)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_xlim(-1.1, 1.1)
    ax.set_ylim(-1.1, 1.1)
    ax.set_aspect("equal")
    ax.set_xlabel("Re")
    ax.set_ylabel("Im")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 5.2 The Bloch sphere

The six basis states sit at the ends of the three axes; the state of Example 2 (red) is below the equator at longitude $-45°$. The green dots show $T^k|+\rangle$ for $k = 0, \dots, 7$: each $T$ advances the relative phase by $\frac\pi4$, i.e. turns the Bloch vector by $45°$ about the $z$-axis, so eight applications walk once around the equator.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def bloch_vector(state):
    alpha, beta = state / np.linalg.norm(state)
    cross = np.conj(alpha) * beta
    return np.array([2 * cross.real, 2 * cross.imag, abs(alpha) ** 2 - abs(beta) ** 2])


s = 1 / np.sqrt(2)
labelled = {
    "$|0\\rangle$": np.array([1, 0]), "$|1\\rangle$": np.array([0, 1]),
    "$|+\\rangle$": np.array([s, s]), "$|-\\rangle$": np.array([s, -s]),
    "$|+i\\rangle$": np.array([s, 1j * s]), "$|-i\\rangle$": np.array([s, -1j * s]),
}
example = np.array([1 + 1j, 2]) / np.sqrt(6)
T = np.diag([1, np.exp(1j * np.pi / 4)])

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(projection="3d")
u, v = np.meshgrid(np.linspace(0, 2 * np.pi, 40), np.linspace(0, np.pi, 20))
ax.plot_wireframe(np.cos(u) * np.sin(v), np.sin(u) * np.sin(v), np.cos(v), color="lightgray", linewidth=0.5)
equator = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(equator), np.sin(equator), 0, color="gray", linewidth=1)
for axis in np.eye(3):
    ax.plot(*zip(-1.2 * axis, 1.2 * axis), color="black", linewidth=0.6)

for label, state in labelled.items():
    r = bloch_vector(state.astype(complex))
    ax.scatter(*r, color="tab:blue", s=40)
    ax.text(*(1.3 * r), label, fontsize=12, ha="center")

r_example = bloch_vector(example)
ax.quiver(0, 0, 0, *r_example, color="tab:red", linewidth=2.5, arrow_length_ratio=0.1)
ax.text(*(1.15 * r_example), "Example 2", color="tab:red")

state = np.array([s, s], dtype=complex)
walk = []
for _ in range(8):
    walk.append(bloch_vector(state))
    state = T @ state
walk = np.array(walk)
ax.scatter(walk[:, 0], walk[:, 1], walk[:, 2], color="tab:green", s=50, depthshade=False, label="$T^k|+\\rangle$, k = 0..7")

ax.set_box_aspect((1, 1, 1))
ax.set_xlim(-1.2, 1.2)
ax.set_ylim(-1.2, 1.2)
ax.set_zlim(-1.2, 1.2)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.view_init(elev=32, azim=-125)
ax.set_title("Bloch sphere")
ax.legend(loc="upper left")
plt.show()

### 5.3 Interference: arrows add before they are squared

Left: the amplitude of outcome $0$ after $H\,R(\phi)\,H$ is the sum of the arrows $\frac12$ (path through $|0\rangle$) and $\frac12e^{i\phi}$ (path through $|1\rangle$). As $\phi$ grows the second arrow turns, and the sum shrinks from length $1$ to $0$. Right: the probability $|\mathrm{sum}|^2 = \cos^2(\phi/2)$, with the frequencies from $1000$ simulated shots, and the flat line $\frac12$ that adding probabilities instead of amplitudes would predict.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, (ax_arrows, ax_prob) = plt.subplots(1, 2, figsize=(14, 5.8), gridspec_kw={"width_ratios": [1, 1.3]})
phis = [0, np.pi / 2, 2 * np.pi / 3, np.pi]
colors = ["tab:blue", "tab:green", "tab:purple", "tab:red"]
for phi, color in zip(phis, colors):
    first, second = 0.5, 0.5 * np.exp(1j * phi)
    total = first + second
    ax_arrows.annotate("", xy=(first, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="black", linewidth=2))
    ax_arrows.annotate("", xy=(total.real, total.imag), xytext=(first, 0), arrowprops=dict(arrowstyle="->", color=color, linewidth=2))
    ax_arrows.plot(total.real, total.imag, "o", color=color, markersize=8,
                   label=f"$\\phi$ = {phi / np.pi:.3g}$\\pi$: |sum| = {abs(total):.3f}")
    ax_arrows.plot([0, total.real], [0, total.imag], ":", color=color)
ax_arrows.plot(0.5 + 0.5 * np.cos(np.linspace(0, 2 * np.pi, 200)), 0.5 * np.sin(np.linspace(0, 2 * np.pi, 200)), color="lightgray")
ax_arrows.axhline(0, color="black", linewidth=0.8)
ax_arrows.axvline(0, color="black", linewidth=0.8)
ax_arrows.set_xlim(-0.2, 1.2)
ax_arrows.set_ylim(-0.25, 0.75)
ax_arrows.set_aspect("equal")
ax_arrows.set_xlabel("Re")
ax_arrows.set_ylabel("Im")
ax_arrows.set_title("amp(0) = $\\frac{1}{2} + \\frac{1}{2}e^{i\\phi}$")
ax_arrows.grid(True, alpha=0.3)
ax_arrows.legend(loc="upper left", fontsize=9)

H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
rng = np.random.default_rng(5)
phi_grid = np.linspace(0, 2 * np.pi, 400)
phi_samples = np.linspace(0, 2 * np.pi, 17)
observed = []
for phi in phi_samples:
    final = H @ np.diag([1, np.exp(1j * phi)]) @ H @ np.array([1, 0], dtype=complex)
    observed.append(np.mean(rng.random(1000) < abs(final[0]) ** 2))

ax_prob.plot(phi_grid, np.cos(phi_grid / 2) ** 2, color="tab:blue", label="$P(0) = \\cos^2(\\phi/2)$")
ax_prob.plot(phi_samples, observed, "o", color="tab:orange", label="1000 simulated shots")
ax_prob.axhline(0.5, color="gray", linestyle="--", label="adding probabilities: 1/2")
ax_prob.set_xticks(np.arange(0, 2.01 * np.pi, np.pi / 2))
ax_prob.set_xticklabels(["0", "$\\pi/2$", "$\\pi$", "$3\\pi/2$", "$2\\pi$"])
ax_prob.set_xlabel("phase $\\phi$")
ax_prob.set_ylabel("probability of outcome 0")
ax_prob.set_title("A phase you cannot see directly controls the outcome")
ax_prob.grid(True, alpha=0.3)
ax_prob.legend(loc="upper center")
plt.tight_layout()
plt.show()

### 5.4 Gates as rotations of the sphere

Start from the Example 2 state and apply $R(\phi)$ for $\phi \in [0, 2\pi]$: the Bloch vector sweeps a circle of constant latitude (left, top view). Apply $H$ to many states on that circle: the whole circle is carried by a half-turn about the $(x + z)/\sqrt2$ axis to a circle around the $x$-axis (right, side view onto the $xz$-plane).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def bloch_vector(state):
    alpha, beta = state / np.linalg.norm(state)
    cross = np.conj(alpha) * beta
    return np.array([2 * cross.real, 2 * cross.imag, abs(alpha) ** 2 - abs(beta) ** 2])


H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
start = np.array([1 + 1j, 2]) / np.sqrt(6)
phis = np.linspace(0, 2 * np.pi, 200)
orbit = np.array([bloch_vector(np.diag([1, np.exp(1j * p)]) @ start) for p in phis])
after_h = np.array([bloch_vector(H @ np.diag([1, np.exp(1j * p)]) @ start) for p in phis])

fig, (ax_top, ax_side) = plt.subplots(1, 2, figsize=(12, 6))
unit = np.exp(1j * np.linspace(0, 2 * np.pi, 200))
for ax in (ax_top, ax_side):
    ax.plot(unit.real, unit.imag, color="lightgray")
    ax.axhline(0, color="black", linewidth=0.6)
    ax.axvline(0, color="black", linewidth=0.6)
    ax.set_aspect("equal")
    ax.set_xlim(-1.25, 1.25)
    ax.set_ylim(-1.25, 1.25)
    ax.grid(True, alpha=0.3)

ax_top.plot(orbit[:, 0], orbit[:, 1], color="tab:red", label="$R(\\phi)$ applied to Example 2")
ax_top.plot(orbit[0, 0], orbit[0, 1], "o", color="tab:red")
ax_top.set_xlabel("x")
ax_top.set_ylabel("y")
ax_top.set_title("Top view: $R(\\phi)$ rotates about the z-axis")
ax_top.legend(loc="lower left", fontsize=9)

ax_side.plot(orbit[:, 0], orbit[:, 2], color="tab:red", label="before H")
ax_side.plot(after_h[:, 0], after_h[:, 2], color="tab:blue", label="after H")
ax_side.plot([-1.2 * np.sqrt(0.5), 1.2 * np.sqrt(0.5)], [-1.2 * np.sqrt(0.5), 1.2 * np.sqrt(0.5)], "--", color="gray", label="H axis $(x+z)/\\sqrt{2}$")
ax_side.set_xlabel("x")
ax_side.set_ylabel("z")
ax_side.set_title("Side view: H is a half-turn swapping x and z")
ax_side.legend(loc="lower right", fontsize=9)
plt.tight_layout()
plt.show()

## 6. Exercises

### Basic

1. Normalize $3|0\rangle + 4i|1\rangle$ and give the measurement probabilities.
2. Which of the following (after normalization) describe the same physical state as $|+\rangle$: $\ -|0\rangle - |1\rangle$, $\ |0\rangle - |1\rangle$, $\ i|0\rangle + i|1\rangle$, $\ e^{i\pi/7}(|0\rangle + |1\rangle)$?
3. Verify by hand that $S$ and $H$ are unitary.

### Intermediate

4. **Draw it first, then compute.** Sketch the Bloch sphere with $|0\rangle$, $|+\rangle$ and $|{+i}\rangle$ marked. Using only "phase gates rotate about the $z$-axis", predict where $S|+\rangle$ and $T^2|+\rangle$ land and where $S|0\rangle$ lands. Then compute the three states.
5. Find the Bloch angles and Bloch vector of $\frac{\sqrt3\,|0\rangle + e^{i\pi/3}|1\rangle}{2}$.
6. For the circuit $H\,R(\phi)\,H$ on $|0\rangle$, find all $\phi \in [0, 2\pi)$ with $P(1) = \frac34$.

### Advanced

7. For $|\psi\rangle = \cos\frac\theta2|0\rangle + e^{i\varphi}\sin\frac\theta2|1\rangle$, show that $P(+) = |\langle +|\psi\rangle|^2 = \frac{1 + \sin\theta\cos\varphi}{2}$. Explain why this shows that the relative phase is measurable while a global phase is not.
8. Show that if $|\psi\rangle = (\alpha, \beta)$ is a unit vector then $|\chi\rangle = (-\bar\beta, \bar\alpha)$ is orthogonal to it, and that its Bloch vector is $-\vec r_\psi$.
9. Show that $R(\phi)$ maps the Bloch vector $(x, y, z)$ to $(x\cos\phi - y\sin\phi,\ x\sin\phi + y\cos\phi,\ z)$, and that $H$ maps it to $(z, -y, x)$. (Use the formula in terms of $\bar\alpha\beta$.)

## 7. Solutions

1. $|3|^2 + |4i|^2 = 25$, so $|\psi\rangle = \frac{3|0\rangle + 4i|1\rangle}{5}$ with $P(0) = \frac{9}{25}$, $P(1) = \frac{16}{25}$.
2. $-|0\rangle - |1\rangle = e^{i\pi}(|0\rangle + |1\rangle)$, $i|0\rangle + i|1\rangle = e^{i\pi/2}(|0\rangle + |1\rangle)$ and $e^{i\pi/7}(|0\rangle + |1\rangle)$ differ from $|+\rangle$ only by a global phase: same state. $|0\rangle - |1\rangle$ has relative phase $\pi$: it is $|-\rangle$, a different (orthogonal) state.
3. $S^\dagger S = \mathrm{diag}(1, \bar i)\,\mathrm{diag}(1, i) = \mathrm{diag}(1, \bar i i) = \mathrm{diag}(1, 1)$. $H$ is real and symmetric, so $H^\dagger H = H^2 = \frac12\begin{pmatrix}2 & 0\\ 0 & 2\end{pmatrix} = I$.
4. Prediction: $S$ turns the sphere by $90°$ about $z$, carrying $+x$ to $+y$, so $S|+\rangle = |{+i}\rangle$; $T^2 = S$ does the same; $|0\rangle$ is on the rotation axis and stays put. Computation: $S|+\rangle = \frac{|0\rangle + i|1\rangle}{\sqrt2} = |{+i}\rangle$; $T^2|+\rangle = \frac{|0\rangle + e^{i\pi/2}|1\rangle}{\sqrt2} = |{+i}\rangle$; $S|0\rangle = |0\rangle$.
5. $\cos\frac\theta2 = \frac{\sqrt3}{2}$ gives $\theta = \frac\pi3$; $\varphi = \frac\pi3$. Bloch vector $\left(\sin\frac\pi3\cos\frac\pi3, \sin\frac\pi3\sin\frac\pi3, \cos\frac\pi3\right) = \left(\frac{\sqrt3}{4}, \frac34, \frac12\right) \approx (0.433, 0.75, 0.5)$.
6. $P(1) = \sin^2\frac\phi2 = \frac34$ gives $\sin\frac\phi2 = \frac{\sqrt3}{2}$ (it is non-negative for $\phi \in [0, 2\pi)$), so $\frac\phi2 = \frac\pi3$ or $\frac{2\pi}{3}$: $\phi = \frac{2\pi}{3}$ or $\frac{4\pi}{3}$.
7. $\langle +|\psi\rangle = \frac{1}{\sqrt2}\left(\cos\frac\theta2 + e^{i\varphi}\sin\frac\theta2\right)$. Its squared modulus is $\frac12\left(\cos^2\frac\theta2 + \sin^2\frac\theta2 + 2\cos\frac\theta2\sin\frac\theta2\cos\varphi\right) = \frac{1 + \sin\theta\cos\varphi}{2}$. It depends on $\varphi$, so different relative phases give different statistics; a global phase multiplies $\langle +|\psi\rangle$ by $e^{i\gamma}$ and drops out of the modulus.
8. $\langle\psi|\chi\rangle = \bar\alpha(-\bar\beta) + \bar\beta\bar\alpha = 0$. For $\chi$: $\overline{(-\bar\beta)}\,\bar\alpha = -\beta\bar\alpha$, so the $x$ and $y$ components are $2\mathrm{Re}(-\bar\alpha\beta)$ and $2\mathrm{Im}(-\bar\alpha\beta)$, the negatives of $\psi$'s; and $|{-\bar\beta}|^2 - |\bar\alpha|^2 = -(|\alpha|^2 - |\beta|^2)$. So $\vec r_\chi = -\vec r_\psi$: orthogonal states are antipodal.
9. $R(\phi)$: $(\alpha, \beta) \mapsto (\alpha, e^{i\phi}\beta)$, so $\bar\alpha\beta \mapsto e^{i\phi}\bar\alpha\beta$: the complex number $x + iy = 2\bar\alpha\beta$ is multiplied by $e^{i\phi}$, a rotation by $\phi$ in the $xy$-plane, while $z$ is unchanged. $H$: $\alpha' = \frac{\alpha + \beta}{\sqrt2}$, $\beta' = \frac{\alpha - \beta}{\sqrt2}$. Then $z' = |\alpha'|^2 - |\beta'|^2 = 2\mathrm{Re}(\bar\alpha\beta) = x$ and $2\bar\alpha'\beta' = |\alpha|^2 - |\beta|^2 + (\bar\beta\alpha - \bar\alpha\beta) = z - 2i\,\mathrm{Im}(\bar\alpha\beta)$, so $x' = z$ and $y' = -y$.

Numerical checks:

In [ ]:
import numpy as np


def bloch_vector(state):
    alpha, beta = state / np.linalg.norm(state)
    cross = np.conj(alpha) * beta
    return np.array([2 * cross.real, 2 * cross.imag, abs(alpha) ** 2 - abs(beta) ** 2])


def same_state(u, v):
    u, v = u / np.linalg.norm(u), v / np.linalg.norm(v)
    return np.isclose(abs(np.vdot(u, v)), 1.0)


H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
S = np.diag([1, 1j])
T = np.diag([1, np.exp(1j * np.pi / 4)])
plus = np.array([1, 1], dtype=complex) / np.sqrt(2)
plus_i = np.array([1, 1j]) / np.sqrt(2)

psi1 = np.array([3, 4j]) / 5
print("Ex 1:", np.round(np.abs(psi1) ** 2, 4))
candidates = {"-|0>-|1>": [-1, -1], "|0>-|1>": [1, -1], "i|0>+i|1>": [1j, 1j], "e^(i pi/7)(|0>+|1>)": np.exp(1j * np.pi / 7) * np.array([1, 1])}
print("Ex 2:", {name: bool(same_state(np.array(v, dtype=complex), plus)) for name, v in candidates.items()})
print("Ex 3:", np.allclose(S.conj().T @ S, np.eye(2)), np.allclose(H.conj().T @ H, np.eye(2)))
print("Ex 4:", same_state(S @ plus, plus_i), same_state(T @ T @ plus, plus_i), same_state(S @ np.array([1, 0]), np.array([1, 0])))
print("Ex 5:", np.round(bloch_vector(np.array([np.sqrt(3), np.exp(1j * np.pi / 3)]) / 2), 4))
for phi in (2 * np.pi / 3, 4 * np.pi / 3):
    final = H @ np.diag([1, np.exp(1j * phi)]) @ H @ np.array([1, 0])
    print(f"Ex 6: phi = {phi:.4f}: P(1) = {abs(final[1]) ** 2:.4f}")

rng = np.random.default_rng(11)
theta, varphi = rng.uniform(0, np.pi), rng.uniform(0, 2 * np.pi)
psi = np.array([np.cos(theta / 2), np.exp(1j * varphi) * np.sin(theta / 2)])
print("Ex 7:", np.isclose(abs(np.vdot(plus, psi)) ** 2, (1 + np.sin(theta) * np.cos(varphi)) / 2))
alpha, beta = psi
chi = np.array([-np.conj(beta), np.conj(alpha)])
print("Ex 8: orthogonal", np.isclose(np.vdot(psi, chi), 0), " antipodal", np.allclose(bloch_vector(chi), -bloch_vector(psi)))
x, y, z = bloch_vector(psi)
phi = 0.7
print("Ex 9: R(phi)", np.allclose(bloch_vector(np.diag([1, np.exp(1j * phi)]) @ psi), [x * np.cos(phi) - y * np.sin(phi), x * np.sin(phi) + y * np.cos(phi), z]),
      "  H", np.allclose(bloch_vector(H @ psi), [z, -y, x]))

## 8. Conclusion

- A qubit is a unit vector $\alpha|0\rangle + \beta|1\rangle$ in $\mathbb{C}^2$; measurement probabilities are squared moduli (Born rule).
- A **global phase** $e^{i\gamma}$ is unobservable; the **relative phase** $\arg\beta - \arg\alpha$ is physical and shows up in measurements in other bases.
- Up to global phase, $|\psi\rangle = \cos\frac\theta2|0\rangle + e^{i\varphi}\sin\frac\theta2|1\rangle$: a point on the **Bloch sphere**, with the relative phase as longitude. Orthogonal states are antipodal, and $\beta/\alpha$ is the stereographic coordinate on the Riemann sphere.
- Gates are **unitary** complex matrices ($X, Y, Z, H, S, T$), which act as rotations of the Bloch sphere; phase gates rotate about the $z$-axis.
- In $H\,R(\phi)\,H$, two amplitudes $\frac12$ and $\frac12e^{i\phi}$ add as arrows before squaring, giving $P(0) = \cos^2\frac\phi2$: a phase that no single measurement reveals decides the outcome. This is interference, and it is the resource every quantum algorithm exploits.

**Next:** this is the last notebook of the **Learning Complex Numbers** series. For further study, continue with the companion [quantum-computing-course](../../../quantum-computing-course/notebooks/00_complex_linear_algebra.ipynb): after its first two notebooks ([complex linear algebra](../../../quantum-computing-course/notebooks/00_complex_linear_algebra.ipynb), [the qubit](../../../quantum-computing-course/notebooks/01_the_qubit.ipynb)), go on to [single-qubit gates](../../../quantum-computing-course/notebooks/02_single_qubit_gates.ipynb), [multiple qubits](../../../quantum-computing-course/notebooks/03_multiple_qubits.ipynb), [interference and oracles](../../../quantum-computing-course/notebooks/05_interference_and_oracles.ipynb) and [the quantum Fourier transform and phase estimation](../../../quantum-computing-course/notebooks/07_qft_and_phase_estimation.ipynb) — where the roots of unity and the DFT of notebook 03 return as a quantum circuit.